# Phase 4 — Strategy backtest

Same rule as the calibration notebook: this notebook **displays** results,
it does not compute its own. Every number and figure comes from
`src/strategy/backtest.py` and `src/strategy/sensitivity.py`, the same code
`make analyze` and `make figures` run.

Method and the reasoning behind each design choice:
`docs/adr/006-train-test-split.md`, `docs/adr/007-strategy-and-sizing.md`.
Honest caveats: `docs/limitations.md`.


In [ ]:
import pandas as pd
from IPython.display import Image, display

from src.config import get_config
from src.strategy.backtest import (
    backtest_anti_bias_control,
    backtest_bias_strategy,
)
from src.strategy.sensitivity import sweep_min_net_edge, sweep_train_fraction, sweep_kelly_fraction

config = get_config()
df = pd.read_parquet(config.clean.processed_path)
print(f"{len(df):,} contracts")

## 1. The split

Chronological, on a contract-count quantile (not a calendar midpoint --
volume ramps 3.5x across the window). Everything the strategy learns from
settled before the boundary; everything it trades is priced at or after it.
The gap between the two conditions is dropped, not straddled.


In [ ]:
ledger, main, split, rules = backtest_bias_strategy(df)
print(f"split at {split.split_ts:%Y-%m-%d}")
print(f"train {len(split.train):,}   test {len(split.test):,}   "
      f"excluded (settle/price gap) {split.excluded}")
rules

## 2. The out-of-sample result, against its own falsification control

The control is the same machinery with every belief inverted -- it buys
overpriced longshots and fades underpriced favorites. If the finding is
real, it must lose money on the same contracts paying the same fees.


In [ ]:
_, control, _, _ = backtest_anti_bias_control(df)
pd.DataFrame({'strategy': main, 'control': control}).T

In [ ]:
display(Image('reports/figures/05_equity_curve.png'))

### Read the breakeven slippage before the ROI

The backtest fills at the last traded price. A real order crosses the
bid-ask spread, and that cost cannot be measured from this data -- settled
snapshots carry no usable order book. `breakeven_slippage` is the adverse
fill per contract that would erase the entire edge.


In [ ]:
print(f"breakeven slippage: {main['breakeven_slippage']*100:.2f}c per contract")
print(f"(compare against a plausible bid-ask half-spread before trusting the ROI)")

In [ ]:
display(Image('reports/figures/06_return_distribution.png'))

## 3. Sensitivity — does the result depend on the exact thresholds chosen?

Three axes swept independently, each re-running the full backtest and its
control. `falsifies` marks configurations where the control still loses
money while the strategy still profits -- the property that matters more
than the exact ROI at any one point.


In [ ]:
sweep_min_net_edge(df)

In [ ]:
sweep_train_fraction(df)

In [ ]:
sweep_kelly_fraction(df)

`kelly_fraction` should leave ROI roughly flat (it rescales stakes, it
doesn't change which side wins) while widening `max_drawdown` -- a sanity
check on the sizing rule rather than a search for a better multiplier.


In [ ]:
display(Image('reports/figures/07_sensitivity.png'))

## 4. Limitations

The headline ROI is a no-spread upper bound, the out-of-sample window is
48 days of a Sports-heavy period, and the underlying bias is small enough
to be sensitive to specification choices made earlier in the pipeline.
Full discussion: `docs/limitations.md`.
